# SENTINEL — Notebook 02: Training + Evaluation
Run for each dataset: HDFS → BGL → OpenStack

**Set DATASET_NAME below before running.**

In [ ]:
import sys
sys.path.insert(0, '/content/SENTINEL')

import torch
import random
import numpy as np

# ── Config ──
DATASET_NAME = 'HDFS'     # Change to 'BGL' or 'OpenStack'
DEV_MODE     = True       # True = 2000 samples, False = full dataset

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE} | Dataset: {DATASET_NAME} | Dev mode: {DEV_MODE}')

In [ ]:
from sentinel.data.parser import load_parsed
from sentinel.data.dataset import split_sessions, get_dataloaders
from sentinel.models.sentinel import SENTINEL
from sentinel.utils.trainer import (
    train_sentinel, collect_scores,
    compute_youden_threshold, compute_metrics,
    print_metrics, plot_training_curves, plot_score_distribution,
)
from configs.config import cfg

# ── Load parsed sessions ──
parsed_paths = {
    'HDFS':      f'{cfg.PARSED_DIR}/hdfs_parsed.pkl',
    'BGL':       f'{cfg.PARSED_DIR}/bgl_parsed.pkl',
    'OpenStack': f'{cfg.PARSED_DIR}/openstack_parsed.pkl',
}
vocab_paths = {
    'HDFS':      f'{cfg.PARSED_DIR}/hdfs_parser.pkl',
    'BGL':       f'{cfg.PARSED_DIR}/bgl_parser.pkl',
    'OpenStack': f'{cfg.PARSED_DIR}/openstack_parser.pkl',
}

obj      = load_parsed(parsed_paths[DATASET_NAME])
sessions = obj['sessions']
labels   = obj['labels']
print(f'Loaded {len(sessions)} sessions')

In [ ]:
# ── Split ──
train_data, val_data, test_data = split_sessions(
    sessions, labels,
    train_ratio = cfg.TRAIN_RATIO,
    val_ratio   = cfg.VAL_RATIO,
    dev_mode    = DEV_MODE,
    dev_samples = cfg.DEV_SAMPLES,
)

# ── DataLoaders ──
train_loader, val_loader, test_loader = get_dataloaders(
    train_data, val_data, test_data,
    max_seq_len = cfg.MAX_SEQ_LEN,
    pad_idx     = cfg.PAD_IDX,
    batch_size  = cfg.BATCH_SIZE,
)

In [ ]:
# ── Determine vocab size from saved parser ──
import pickle
with open(vocab_paths[DATASET_NAME], 'rb') as f:
    vocab_obj = pickle.load(f)
VOCAB_SIZE = vocab_obj['next_id'] + 10   # small buffer
print(f'Vocab size: {VOCAB_SIZE}')

# ── Build Model ──
model = SENTINEL(
    vocab_size         = VOCAB_SIZE,
    embed_dim          = cfg.EMBED_DIM,
    pad_idx            = cfg.PAD_IDX,
    transformer_layers = cfg.TRANSFORMER_LAYERS,
    num_heads          = cfg.ATTENTION_HEADS,
    ffn_dim            = cfg.FFN_DIM,
    gnn_hidden         = cfg.GNN_HIDDEN,
    gnn_layers         = cfg.GNN_LAYERS,
    latent_dim         = cfg.FUSION_OUTPUT,
    flow_layers        = cfg.FLOW_LAYERS,
    flow_hidden        = cfg.FLOW_HIDDEN,
    dropout            = cfg.DROPOUT,
)
print(f'SENTINEL parameters: {model.count_parameters():,}')

In [ ]:
# ── Train ──
model, train_losses, val_losses = train_sentinel(
    model, train_loader, val_loader,
    epochs        = cfg.EPOCHS,
    lr            = cfg.LEARNING_RATE,
    weight_decay  = cfg.WEIGHT_DECAY,
    patience      = cfg.PATIENCE,
    use_amp       = cfg.USE_AMP,
    device        = DEVICE,
    checkpoint_dir = f'{cfg.RESULTS_DIR}/checkpoints',
    dataset_name  = DATASET_NAME,
)

plot_training_curves(
    train_losses, val_losses,
    save_path    = f'{cfg.RESULTS_DIR}/{DATASET_NAME}_train_curve.png',
    dataset_name = DATASET_NAME,
)

In [ ]:
# ── YOBDM Threshold on Validation Set ──
val_scores, val_labels = collect_scores(model, val_loader, DEVICE)
threshold = compute_youden_threshold(val_scores, val_labels)

val_metrics = compute_metrics(val_scores, val_labels, threshold)
print_metrics(val_metrics, split='Validation')

In [ ]:
# ── Test Evaluation ──
test_scores, test_labels = collect_scores(model, test_loader, DEVICE)
test_metrics = compute_metrics(test_scores, test_labels, threshold)
print_metrics(test_metrics, split='Test')

plot_score_distribution(
    test_scores, test_labels, threshold,
    save_path    = f'{cfg.RESULTS_DIR}/{DATASET_NAME}_score_dist.png',
    dataset_name = DATASET_NAME,
)

# Save results
import json, os
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f'{cfg.RESULTS_DIR}/{DATASET_NAME}_results.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)
print(f'Results saved → {cfg.RESULTS_DIR}/{DATASET_NAME}_results.json')